In [1]:
import os

BASE = "/kaggle/input/competitions/2026task-4/segmentation" 
TRAIN_IMG  = os.path.join(BASE, "train", "image")
TRAIN_MASK = os.path.join(BASE, "train", "label")
TEST_IMG   = os.path.join(BASE, "test", "image")

# 诊断
for name, p in [("TRAIN_IMG", TRAIN_IMG), ("TRAIN_MASK", TRAIN_MASK), ("TEST_IMG", TEST_IMG)]:
    if os.path.isdir(p):
        fs = [f for f in os.listdir(p) if f.lower().endswith(('.jpg','.png'))]
        print(f"{name}: exists=True, imgs={len(fs)}, sample={fs[:3]}")
    else:
        print(f"{name}: 路径不存在 -> {p}")

TRAIN_IMG: exists=True, imgs=20, sample=['20.jpg', '6.jpg', '5.jpg']
TRAIN_MASK: exists=True, imgs=20, sample=['20.jpg', '6.jpg', '5.jpg']
TEST_IMG: exists=True, imgs=20, sample=['20.jpg', '6.jpg', '5.jpg']


In [2]:
import cv2, numpy as np, torch
from torch.utils.data import Dataset
import albumentations as A
from albumentations.pytorch import ToTensorV2

class RetinaDataset(Dataset):
    def __init__(self, img_dir, mask_dir=None, img_size=512, train=True):
        self.img_dir  = img_dir
        self.mask_dir = mask_dir
        self.img_size = img_size
        self.train    = train

        img_files = {os.path.splitext(f)[0]: f
                     for f in os.listdir(img_dir)
                     if f.lower().endswith(('.jpg','.png'))}

        if train:
            mask_files = {os.path.splitext(f)[0]: f
                          for f in os.listdir(mask_dir)
                          if f.lower().endswith(('.jpg','.png'))}
            # 按文件名(去后缀)配对，只保留两边都有的
            common = sorted(set(img_files) & set(mask_files))
            missing = set(img_files) - set(mask_files)
            if missing:
                print(f"⚠️ {len(missing)} 张图没有对应标签，已跳过：{list(missing)[:5]}")
            self.pairs = [(img_files[k], mask_files[k]) for k in common]
        else:
            self.pairs = [(f, None) for f in sorted(img_files.values())]

        assert len(self.pairs) > 0, f"配对后为0，请检查 {img_dir} 与 {mask_dir} 的命名"

        if train:
            self.tf = A.Compose([
                A.Resize(img_size, img_size),
                A.HorizontalFlip(p=0.5),
                A.VerticalFlip(p=0.5),
                A.RandomRotate90(p=0.5),
                A.RandomBrightnessContrast(p=0.3),
                A.ElasticTransform(p=0.2),
                A.Normalize(mean=(0.485,0.456,0.406),
                            std=(0.229,0.224,0.225)),
                ToTensorV2(),
            ])
        else:
            self.tf = A.Compose([
                A.Resize(img_size, img_size),
                A.Normalize(mean=(0.485,0.456,0.406),
                            std=(0.229,0.224,0.225)),
                ToTensorV2(),
            ])

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        img_name, mask_name = self.pairs[idx]
        img = cv2.cvtColor(cv2.imread(os.path.join(self.img_dir, img_name)),
                           cv2.COLOR_BGR2RGB)

        if self.train:
            mask = cv2.imread(os.path.join(self.mask_dir, mask_name),
                              cv2.IMREAD_GRAYSCALE)
            mask = (mask > 127).astype(np.float32)   # 血管=1，背景=0
            aug = self.tf(image=img, mask=mask)
            return aug['image'], aug['mask'].unsqueeze(0)

        # 测试：返回原图尺寸用于把预测还原回去
        # 测试分支：只返回 2 个值
        aug = self.tf(image=img)
        return aug['image'], img_name

In [3]:
from torch.utils.data import DataLoader

train_ds = RetinaDataset(TRAIN_IMG, mask_dir=TRAIN_MASK, train=True)
print("train size:", len(train_ds))          # 应为 20

train_loader = DataLoader(train_ds, batch_size=4, shuffle=True,
                          num_workers=2, pin_memory=True)

# 看一眼一个 batch 是否正常
imgs, masks = next(iter(train_loader))
print("imgs:", imgs.shape, "masks:", masks.shape,
      "mask 唯一值:", torch.unique(masks))

train size: 20
imgs: torch.Size([4, 3, 512, 512]) masks: torch.Size([4, 1, 512, 512]) mask 唯一值: tensor([0., 1.])


In [4]:
!pip install -q segmentation-models-pytorch albumentations

In [5]:
import segmentation_models_pytorch as smp
import torch.nn as nn
from torch.optim import Adam

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

model = smp.Unet(
    encoder_name="resnet34",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1,
).to(DEVICE)

# Dice + BCE 混合损失
bce = nn.BCEWithLogitsLoss()
def dice_loss(logits, target, eps=1e-6):
    prob = torch.sigmoid(logits)
    inter = (prob*target).sum(dim=(2,3))
    union = prob.sum(dim=(2,3)) + target.sum(dim=(2,3))
    return 1 - ((2*inter + eps)/(union + eps)).mean()

def loss_fn(logits, target):
    return 0.5*bce(logits, target) + 0.5*dice_loss(logits, target)

config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 87.3MB            

model.safetensors: downloading bytes:           |  0.00B            

In [6]:
def train_one_epoch(model, loader, opt):
    model.train()
    total = 0
    for imgs, masks in loader:
        imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
        opt.zero_grad()
        logits = model(imgs)
        loss = loss_fn(logits, masks)
        loss.backward()
        opt.step()
        total += loss.item()*imgs.size(0)
    return total/len(loader.dataset)
#训练循环
train_ds = RetinaDataset(TRAIN_IMG, mask_dir=os.path.join(BASE,"train","label"), train=True)
train_loader = DataLoader(train_ds, batch_size=4, shuffle=True, num_workers=2)

opt = Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=50)

for epoch in range(30):
    loss = train_one_epoch(model, train_loader, opt)
    scheduler.step()
    print(f"Epoch {epoch+1}: loss={loss:.4f}")

Epoch 1: loss=0.6801
Epoch 2: loss=0.5344
Epoch 3: loss=0.4594
Epoch 4: loss=0.4038
Epoch 5: loss=0.3633
Epoch 6: loss=0.3285
Epoch 7: loss=0.3029
Epoch 8: loss=0.2815
Epoch 9: loss=0.2642
Epoch 10: loss=0.2504
Epoch 11: loss=0.2379
Epoch 12: loss=0.2317
Epoch 13: loss=0.2242
Epoch 14: loss=0.2179
Epoch 15: loss=0.2122
Epoch 16: loss=0.2096
Epoch 17: loss=0.2101
Epoch 18: loss=0.2058
Epoch 19: loss=0.2029
Epoch 20: loss=0.1988
Epoch 21: loss=0.1951
Epoch 22: loss=0.1952
Epoch 23: loss=0.1915
Epoch 24: loss=0.1912
Epoch 25: loss=0.1913
Epoch 26: loss=0.1911
Epoch 27: loss=0.1892
Epoch 28: loss=0.1861
Epoch 29: loss=0.1859
Epoch 30: loss=0.1850


In [7]:
from torch.utils.data import DataLoader

test_ds = RetinaDataset(TEST_IMG, train=False, img_size=512)
print("test size:", len(test_ds))
test_loader = DataLoader(test_ds, batch_size=1, shuffle=False, num_workers=2)

test size: 20


In [8]:
import os, cv2, numpy as np, torch

PRED_DIR = "image"          # 与最后 cell 的 './image' 对接
os.makedirs(PRED_DIR, exist_ok=True)
model.eval()

with torch.no_grad():
    for imgs, fnames in test_loader:          # Dataset 测试分支只返回 2 值
        imgs = imgs.to(DEVICE)
        logits = model(imgs)
        prob = torch.sigmoid(logits)[0, 0].cpu().numpy()

        orig = cv2.imread(os.path.join(TEST_IMG, fnames[0]), cv2.IMREAD_GRAYSCALE)
        h, w = orig.shape[:2]
        prob = cv2.resize(prob, (w, h), interpolation=cv2.INTER_LINEAR)

        pred = (prob > 0.5).astype(np.uint8)
        out = np.where(pred == 1, 255, 0).astype(np.uint8)   # 匹配最后 cell 的 >0 逻辑
        cv2.imwrite(os.path.join(PRED_DIR, fnames[0]), out)

print("done, files:", len(os.listdir(PRED_DIR)))

done, files: 20


In [9]:
import numpy as np
import pandas as pd
from PIL import Image
import torchvision
import os
"""
In order to reduce the submission file size, our metric uses run-length encoding on the pixel values.
Instead of submitting an exhaustive list of indices for your segmentation, you will submit pairs of values 
that contain a start position and a run length.
E.g. '1 3' implies starting at pixel 1 and running a total of 3 pixels 1,2,3. The competition format requires a space
delimited list of pairs. For example, '1 3 10 5' implies pixels 1,2,3,10,11,12,13,14 are to be included in the mask. 
The metric checks that the pairs are sorted, positive, and the decoded pixel values are not duplicated. 
----------------------------------------------------------------------------------------------------------------
Put the predicted segmentation images into the "image" folder(You need to create one) and run this python file.
"""

def get_img_file(image_dir):
    imagelist = []
    namelist = []
    for parent, dirnames, filenames in os.walk(image_dir):
        for filename in filenames:
            if filename.lower().endswith(
                    ('.bmp', '.dib', '.png', '.jpg', '.jpeg', '.pbm', '.pgm', '.ppm', '.tif', '.gif')):
                imagelist.append(os.path.join(parent, filename))
                namelist.append(filename)
        return imagelist, namelist


def turn_to_str(image_list):
    outputs = []
    for image_path in image_list:
        image = Image.open(image_path).convert('L')
        transform = torchvision.transforms.ToTensor()
        image = image.resize((512, 512), Image.Resampling.BILINEAR)
        image = transform(image)
        image[image > 0] = 1
        dots = np.where(image.flatten() == 1)[0]
        run_lengths = []
        prev = -2
        for b in dots:
            if (b > prev + 1):
                run_lengths.extend((b + 1, 0))
            run_lengths[-1] += 1
            prev = b
        output = ' '.join([str(r) for r in run_lengths])
        outputs.append(output)
    return outputs


def save_to_csv(name_list, str_list):
    df = pd.DataFrame(columns=['Id', 'Predicted'])
    df['Id'] = [i.split('.')[0] for i in name_list]
    df['Predicted'] = str_list
    df.to_csv('submission.csv', index=None)


if __name__=="__main__":
    image_dir = './image'
    image_list, name_list = get_img_file(image_dir)
    str_list = turn_to_str(image_list)
    save_to_csv(name_list, str_list)